# W05B 실습 B · MF 구현, 학습 곡선과 웹 앱

**딥러닝응용I(추천시스템) · 동덕여자대학교 · 유원상 교수 · 2026-2**

2026년 10월 1일 · W05B · 주교재 4.1–4.3.
목표: SGD를 반복하는 MF를 실행하고 학습 결과와 한계를 설명합니다.
CPU 런타임으로 실행합니다. 파일 → Drive에 사본 저장 후 첫 셀부터 실행하세요.
설치 후 재시작 안내가 나오면 런타임을 재시작하고 첫 셀부터 실행합니다.
None과 빈 문자열은 연습용입니다. 미완성 문제를 건너뛰어도 독립 시연은 실행됩니다.
자동 검사가 통과해도 자신의 빈칸 답이 모두 맞았다는 뜻은 아닙니다.


In [ ]:
import os,sys,subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w05b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from luna_recsys.mf_sgd import MFSGD,sgd_step
from luna_recsys.mf_lab import step_view,training_view,build_mf_app
from luna_recsys.model_comparison import comparison_toy,definition_link
print('Python',sys.version.split()[0],'수업 버전',COURSE_VERSION)


## 함수 설명과 정의 바로가기
설치한 버전의 정의 첫 줄로 연결됩니다. `MFSGD(...)`는 설정만 저장하고
`fit(관측표)`이 초기화부터 다시 학습합니다. `history_`는 epoch·train_rmse·objective 표입니다.
`predict(ID,ID)`는 실수, `explain(ID,ID)`은 평균·편향·내적·알려진 ID 여부 사전,
`recommend(ID,N)`은 이미 본 영화를 제외한 표를 반환합니다.
`sgd_step(p,q,rating,...)`는 원본을 바꾸지 않고 갱신 전후 사전을 반환합니다.
`build_mf_app(관측표)`는 화면만 만들며 `launch`는 별도입니다.


In [ ]:
apis=[MFSGD,MFSGD.fit,MFSGD.predict,MFSGD.explain,MFSGD.rmse,MFSGD.recommend,
      sgd_step,step_view,training_view,build_mf_app,comparison_toy,definition_link]
display(Markdown('\n'.join(f'- [{obj.__qualname__}]({definition_link(obj,COURSE_VERSION)})' for obj in apis)))
toy=comparison_toy()
ratings=toy.ratings.copy()
print('작은 예제 관측 수:',len(ratings))
display(ratings.pivot(index='user_id',columns='movie_id',values='rating'))


## 1. 작은 관측표로 클래스의 상태 읽기
생성→fit→history→explain→recommend 순서로 읽습니다.
fit은 이전 학습을 이어 가지 않고 같은 seed에서 새로 학습합니다.


In [ ]:
model=MFSGD(n_factors=2,learning_rate=.02,regularization=.02,epochs=80).fit(ratings)
display(model.history_.iloc[[0,-1]])
print('P,Q 모양:',model.P_.shape,model.Q_.shape)
display(model.explain(1,4))
display(model.recommend(1,5))


In [ ]:
assert model.P_.shape==(5,2) and model.Q_.shape==(5,2)
assert len(model.history_)==81
assert model.history_.train_rmse.iloc[-1]<model.history_.train_rmse.iloc[0]
assert np.isclose(model.mean_,56/18)
seen=set(ratings.loc[ratings.user_id==1,'movie_id'])
assert not seen.intersection(model.recommend(1,5).movie_id)


### B1 · 설명하기: 80 epoch는 무엇을 80번 했는가?
총 갱신 횟수와 epoch 0을 포함한 기록표 행 수를 설명하세요.
힌트: 한 갱신에서 모든 사용자 벡터를 동시에 바꾸지는 않습니다.


In [ ]:
b1_updates=None
b1_reason=''


## 2. 초기화 반례와 한 요소 실험
P와 Q를 0으로 시작하면 서로를 곱하는 기울기가0입니다. 편향은 움직일 수 있습니다.
K를 바꿀 때 초기 표준편차까지1/K로 바꾸면 두 조건을 함께 바꾼 것입니다.
이번 구현은 초기 표준편차0.1을 K와 독립적으로 고정합니다.


In [ ]:
zero=MFSGD(n_factors=2,epochs=20,init_scale=0).fit(ratings)
print('모든 P,Q가0인가:',np.all(zero.P_==0),np.all(zero.Q_==0))
print('편향 변화:',np.linalg.norm(zero.user_bias_))
fig,ax=plt.subplots(figsize=(7,3))
for lr in [.002,.02,.1]:
    trial=MFSGD(n_factors=2,learning_rate=lr,regularization=.02,epochs=80).fit(ratings)
    ax.plot(trial.history_.epoch,trial.history_.train_rmse,label=f'alpha={lr}')
ax.set(xlabel='Epoch',ylabel='Train RMSE');ax.legend();ax.grid(alpha=.2)
display(fig);plt.close(fig)


In [ ]:
assert np.all(zero.P_==0) and np.all(zero.Q_==0)
assert np.linalg.norm(zero.user_bias_)>0


### B2 · 비교 실험
위 루프에서 학습률.02를 고정하고 정규화만0,.02,.5로 바꾸는 코드를 작성하세요.
결과로 설명할 수 있는 것과 알 수 없는 것을 각각 적으세요.
자가 점검: 관측표·seed·K·epoch·초기화 표준편차를 유지했나요?


In [ ]:
b2_code=''
b2_observation=''
b2_limit=''


## 3. 공식 MovieLens 자료를 자동 준비하기
`prepare_movielens(cache_dir,mode="real")`은 검증된 캐시를 쓰거나 공식 ZIP을 내려받습니다.
반환 객체 `data.ratings`는 관측표, `data.movies`는 제목과 장르 표입니다.
실패하면 오류를 보여 줍니다. 합성자료를 실데이터 결과로 바꾸어 표시하지 않습니다.
CPU 실습은10만 관측에서 고정 seed로 뽑은2만 관측을12 epoch 학습합니다.
이 표본은 학습 속도를 위한 것이며 독립 test가 아닙니다.


In [ ]:
from luna_recsys.datasets import prepare_movielens
display(Markdown(f'[prepare_movielens 정의]({definition_link(prepare_movielens,COURSE_VERSION)})'))
prepared=prepare_movielens('data/local/w05b',mode='real')
data=prepared.data
fast=data.ratings.sample(20000,random_state=20261001).reset_index(drop=True)
print(prepared.description,'원본 관측:',len(data.ratings),'학습 관측:',len(fast))
real_model=MFSGD(n_factors=8,learning_rate=.02,regularization=.02,epochs=12).fit(fast)
display(real_model.history_)
fig,ax=plt.subplots(figsize=(7,3))
ax.plot(real_model.history_.epoch,real_model.history_.train_rmse,marker='.')
ax.set(xlabel='Epoch',ylabel='Train RMSE');ax.grid(alpha=.2)
display(fig);plt.close(fig)
user=fast.user_id.iloc[0]
display(real_model.recommend(user,5).merge(data.movies[['movie_id','title']],on='movie_id',how='left'))
print('학습 관측 RMSE:',real_model.rmse(fast))
print('알려지지 않은 사용자:',real_model.explain(-999,fast.movie_id.iloc[0]))


In [ ]:
assert len(data.ratings)==100000 and len(fast)==20000
assert np.isclose(real_model.history_.train_rmse.iloc[0],1.125559542801612,atol=1e-9)
assert np.isclose(real_model.history_.train_rmse.iloc[-1],.7939861560860365,atol=1e-8)
assert np.isclose(real_model.rmse(fast),real_model.history_.train_rmse.iloc[-1])
assert not real_model.explain(-999,fast.movie_id.iloc[0])['known_user']


### B3 · 잘못된 결론 고치기
“MF의 RMSE가0.794이므로 지난 수업의 CF보다 좋고, 다음 추천도 더 정확하다.”
이 문장을 현재 결과가 뒷받침하는 수준으로 고치세요.
힌트: 평가 관측·후보·지표·분할 조건을 먼저 맞춰야 합니다.


In [ ]:
b3_corrected=''


## 4. 웹 앱: 입력→인자→반환→화면
첫 탭은 손계산 상태의 한 단계, 둘째 탭은 관측표 전체 반복입니다.
실제 데이터 앱은 응답 속도를 위해 최대3,000관측을 고정 표본으로 사용합니다.
따라서 위2만 관측 실험과 수치가 같을 필요가 없습니다.
`step_view`는 요약·표·그림, `training_view`는 요약·기록·그림·추천표를 반환합니다.


In [ ]:
message,update_table,update_figure=step_view()
assert len(update_table)==6 and '3.088038' in message
summary,history,curve,ranking=training_view(ratings,2,.02,.02,10,1)
assert len(history)==11 and len(ranking)<=5
display(update_table,update_figure,history.tail(2),curve)


### B4 · 직접 수정하기: 내 설명을 화면에 표시하기
callback은 결과 요약에 `my_observation`을 붙입니다.
p2가 감소하는 이유를 자신의 말로 적고 셀을 다시 실행하세요.
입력 컴포넌트 순서가 함수 인자와 같은지 확인한 뒤 버튼으로 동작을 확인하세요.
자가 점검: 입력을 바꾸면 표가 바뀌고 내가 쓴 설명은 요약 끝에 유지되나요?


In [ ]:
import gradio as gr
my_observation=''  # 자신의 설명을 작성하세요.
def my_step_view(rating,learning_rate,regularization):
    message,table,figure=step_view(rating,learning_rate,regularization)
    note=my_observation.strip() or '내 설명을 작성하고 이 셀을 다시 실행하세요.'
    return message+'\n나의 해석: '+note,table,figure
with gr.Blocks(title='내가 수정한 SGD 설명',analytics_enabled=False) as edited_app:
    gr.Markdown('## 나의 설명을 포함한 SGD 한 단계')
    edit_r=gr.Slider(1,5,value=4,step=1,label='실제 평점')
    edit_lr=gr.Slider(.001,.5,value=.05,step=.001,label='학습률')
    edit_reg=gr.Slider(0,1,value=.02,step=.01,label='정규화')
    edit_button=gr.Button('내 설명과 함께 계산')
    edit_message=gr.Textbox(label='나의 해석')
    edit_table=gr.Dataframe(label='갱신표')
    edit_plot=gr.Plot(label='기여도')
    edit_button.click(my_step_view,[edit_r,edit_lr,edit_reg],
                      [edit_message,edit_table,edit_plot],api_name='my_step')
if IN_COLAB:
    edited_app.launch(share=True,debug=False,prevent_thread_lock=True)
else:
    print('Colab에서 위 셀을 실행하면 수정 앱이 열립니다.')
assert '나의 해석:' in my_step_view(4,.05,.02)[0]


## 5. 마지막 종합 앱과 결과 기록
첫 탭에서 손계산 수치를 재현하고 둘째 탭에서 한 설정만 바꾸어 비교하세요.
기록 양식: 고정 조건 / 바꾼 설정 / 예상 / 실제 수치 / 이유 / 알 수 없는 것.
Colab의 공유 링크를 새 탭으로 열 수도 있습니다. 런타임을 종료하면 링크도 만료됩니다.
앱이 안 보이면 셀 오류와 설치 후 재시작 안내를 확인하세요.


In [ ]:
app=build_mf_app(fast)
if IN_COLAB:
    app.launch(share=True,debug=False,prevent_thread_lock=True)
else:
    print('화면 생성 완료. Colab에서는 공유 앱이 실행됩니다. 로컬에서는 app.launch()로 실행할 수 있습니다.')
assert len(app.config['dependencies'])==2


## 점검 퀴즈
1. 미관측 평점을 0으로 채우면 어떤 가정이 추가되는가?
2. 사용자 5명, 영화 5편, K=2일 때 P와 Q의 모양은?
3. 잔차가 양수여도 p의 어떤 좌표는 왜 감소하는가?
4. 갱신된 P를 Q 갱신에 사용하면 같은 시점 기울기와 무엇이 달라지는가?
5. 학습 RMSE와 정규화 목적함수는 왜 값이 다른가?
6. train RMSE 감소만으로 실제 추천 품질 향상을 결론 내릴 수 있는가?

## 핵심 정리와 출처
관측만 학습하고, 같은 시점의 기울기로 모수를 갱신하고, 결과의 평가 범위를 명시합니다.
임일, 주교재 4.1–4.3. 예제·NumPy 구현·앱은 수업용 독립 제작입니다.
[Google MF](https://developers.google.com/machine-learning/recommendation/collaborative/matrix)
· [Surprise MF](https://surprise.readthedocs.io/en/stable/matrix_factorization.html)
· [GroupLens README](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt)
· [Gradio Blocks](https://www.gradio.app/docs/gradio/blocks)
